In [35]:
import numpy as np
import pandas as pd

In [36]:
def detect_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask = (series < lower) | (series > upper)

    return mask, (lower, upper)

In [37]:
def missing_summary(df):
    return pd.DataFrame({
        "missing_count": df.isnull().sum(),
        "missing_ratio(%)": (df.isnull().mean() * 100).round(2)
    })

In [38]:
# 새 데이터셋 — '옷장패션' 주문 (가상)
np.random.seed(11)
n = 1500

partner = pd.DataFrame({
    "order_id": [f"K{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(33, 8, n).round().astype(int),
    "category": np.random.choice(["상의", "하의", "신발", "액세서리"], n, p=[0.35, 0.3, 0.2, 0.15]),
    "channel": np.random.choice(["web", "app"], n, p=[0.4, 0.6]),
    "price": np.random.choice([15900, 29900, 49900, 79900, 129900], n),
    "quantity": np.random.choice([1, 1, 1, 2, 2, 3], n),
})
partner["amount"] = partner["price"] * partner["quantity"]
partner["return_amount"] = np.where(
    np.random.rand(n) < 0.07, partner["amount"] * np.random.uniform(0.5, 1.0, n), 0
).round(0)

# 오염 심기
# (a) 나이 이상치 — 입력 실수(0, 999)
partner.loc[partner.sample(3, random_state=1).index, "customer_age"] = 999
partner.loc[partner.sample(2, random_state=2).index, "customer_age"] = 0

# (b) amount 결측 — app 채널에 더 자주 (MAR 시그널)
app = partner["channel"] == "app"
partner.loc[partner[app].sample(frac=0.05, random_state=3).index, "amount"] = np.nan
partner.loc[partner[~app].sample(frac=0.01, random_state=4).index, "amount"] = np.nan

# (c) return_amount 결측은 그대로 (0=환불없음)이라 결측 아님. 단, '관찰 안 됨'을 의도적으로 표현하기 위해
#     price 결측 5건 추가(접속 시점 가격이 누락된 사례)
partner.loc[partner.sample(5, random_state=5).index, "price"] = np.nan

# (d) quantity 이상치(단일 소비자 200개)
partner.loc[partner.sample(1, random_state=6).index, "quantity"] = 200

# (e) amount 극단값(50,000,000짜리 한 건 — '도매 의심')
partner.loc[partner.sample(1, random_state=7).index, "amount"] = 50_000_000

print("옷장패션 데이터 준비 완료:", partner.shape)
partner.head()

옷장패션 데이터 준비 완료: (1500, 8)


,order_id,customer_age,category,channel,price,quantity,amount,return_amount
0,K00001,47,신발,app,29900.0,2,59800.0,45445.0
1,K00002,31,상의,app,129900.0,3,389700.0,0.0
2,K00003,29,상의,web,49900.0,2,99800.0,0.0
3,K00004,12,상의,web,49900.0,3,149700.0,0.0
4,K00005,33,하의,app,129900.0,1,129900.0,0.0


In [39]:
print("[열별 결측]")

display(pd.DataFrame({
    "missing_count": partner.isnull().sum(),
    "missing_ratio(%)": (partner.isnull().mean() * 100).round(2)
}))

[열별 결측]


,missing_count,missing_ratio(%)
order_id,0,0.00
customer_age,0,0.00
category,0,0.00
channel,0,0.00
price,5,0.33
quantity,0,0.00
amount,51,3.40
return_amount,0,0.00


In [40]:
# 시나리오 1 — 진단
print("shape:", partner.shape)
partner.info()
display(partner.describe())

shape: (1500, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       1500 non-null   str    
 1   customer_age   1500 non-null   int64  
 2   category       1500 non-null   str    
 3   channel        1500 non-null   str    
 4   price          1495 non-null   float64
 5   quantity       1500 non-null   int64  
 6   amount         1449 non-null   float64
 7   return_amount  1500 non-null   float64
dtypes: float64(3), int64(2), str(3)
memory usage: 93.9 KB


,customer_age,price,quantity,amount,return_amount
count,1500.000000,1495.000000,1500.000000,1.449000e+03,1500.000000
mean,34.903333,60960.869565,1.789333,1.350475e+05,6381.010000
std,43.936525,40275.103681,5.173406,1.313695e+06,28315.037316
min,0.000000,15900.000000,1.000000,1.590000e+04,0.000000
25%,28.000000,29900.000000,1.000000,4.770000e+04,0.000000
50%,33.000000,49900.000000,2.000000,7.990000e+04,0.000000
75%,38.250000,79900.000000,2.000000,1.299000e+05,0.000000
max,999.000000,129900.000000,200.000000,5.000000e+07,322778.000000


# 시나리오 1 진단 결과

- 데이터 크기: 1500행 × 8열

- 결측
  - amount: 약 ○%
  - price: 5건

- 이상치(IQR)


  - customer_age: 999, 0 발견
  - quantity: 200 발견
  - amount: 50,000,000 발견

- amount 결측은 app 채널에 더 많이 발생하여 MAR로 의심된다.

In [41]:
# 결측 진단
print("[열별 결측]")
display(missing_summary(partner))

# 결측이 채널과 관련 있는지 (MAR 신호 검사)
amt_null = partner[partner["amount"].isnull()]
print("\n[amount 결측 행의 채널 분포]")
print(amt_null["channel"].value_counts(normalize=True).round(2))
print("\n[전체 채널 분포]")
print(partner["channel"].value_counts(normalize=True).round(2))

[열별 결측]


,missing_count,missing_ratio(%)
order_id,0,0.00
customer_age,0,0.00
category,0,0.00
channel,0,0.00
price,5,0.33
quantity,0,0.00
amount,51,3.40
return_amount,0,0.00



[amount 결측 행의 채널 분포]
channel
app    0.88
web    0.12
Name: proportion, dtype: float64

[전체 채널 분포]
channel
app    0.61
web    0.39
Name: proportion, dtype: float64


## 결측치 진단

- `amount` 컬럼에서 결측치가 가장 많이 발견되었다.
- `price` 컬럼에서도 일부 결측치가 존재한다.
- 결측치가 발생한 원인을 추가로 확인하여 적절한 처리 방법을 결정할 필요가 있다.

In [42]:
# IQR 이상치 — 수치형 컬럼 일괄 점검
num_cols = ["customer_age", "price", "quantity", "amount", "return_amount"]
print("[IQR 기준 이상치 개수]")
for c in num_cols:
    mask, (lo, up) = detect_outliers_iqr(partner[c].dropna())
    print(f"  {c:15s}  하한={lo:>12.1f}  상한={up:>12.1f}  이상치={mask.sum()}건")

[IQR 기준 이상치 개수]
  customer_age     하한=        12.6  상한=        53.6  이상치=18건
  price            하한=    -45100.0  상한=    154900.0  이상치=0건
  quantity         하한=        -0.5  상한=         3.5  이상치=1건
  amount           하한=    -75600.0  상한=    253200.0  이상치=145건
  return_amount    하한=         0.0  상한=         0.0  이상치=122건


## IQR 기준 이상치 진단

- `customer_age`에서는 IQR 기준으로 18건의 이상치가 발견되었다. 이 중 0세와 999세는 실제 고객 나이로 보기 어려운 입력 오류이므로 결측치로 변환한 뒤 대체할 필요가 있다.
- `price`에서는 IQR 기준 이상치가 발견되지 않았다.
- `quantity`에서는 1건의 이상치가 발견되었다. 수량 200개는 일반적인 개인 주문으로 보기 어려워 입력 오류 또는 도매 주문 가능성을 추가로 확인해야 한다.
- `amount`에서는 145건의 이상치가 발견되었다. 다만 주문금액은 가격과 수량에 따라 실제로 크게 나타날 수 있으므로 IQR 기준만으로 모두 제거하면 안 된다.
- `return_amount`는 대부분의 값이 0이어서 IQR의 하한과 상한이 모두 0으로 계산되었다. 따라서 0보다 큰 정상 환불 금액도 모두 이상치로 분류되므로, 이 컬럼에는 IQR 기준을 그대로 적용하기 어렵다.

In [43]:
# 시나리오 3 — 처리 코드 (예시 구현)
partner_clean = partner.copy()

# 1) customer_age 물리적 불가능 값 → NaN → 중앙값 대체
unrealistic = (partner_clean["customer_age"] < 1) | (partner_clean["customer_age"] > 110)
partner_clean.loc[unrealistic, "customer_age"] = np.nan
partner_clean["customer_age"] = partner_clean["customer_age"].fillna(
    partner_clean["customer_age"].median()
).astype(int)

# 2) quantity 이상치 → NaN → 중앙값 대체
mask_q, _ = detect_outliers_iqr(partner_clean["quantity"])
partner_clean.loc[mask_q, "quantity"] = np.nan
partner_clean["quantity"] = partner_clean["quantity"].fillna(
    partner_clean["quantity"].median()
).astype(int)

# 3) amount 이상치(50,000,000) → 유지 + 플래그
mask_a, _ = detect_outliers_iqr(partner_clean["amount"])
partner_clean["amount_outlier"] = mask_a.astype(int)

# 4) amount 결측 → 채널별 중앙값 대체 (MAR 가설)
partner_clean["amount"] = partner_clean["amount"].fillna(
    partner_clean.groupby("channel")["amount"].transform("median")
)

# 5) price 결측 → 카테고리별 중앙값 대체
partner_clean["price"] = partner_clean["price"].fillna(
    partner_clean.groupby("category")["price"].transform("median")
)

# 검증 출력
print("[처리 전 후 결측 비교]")
before = partner.isnull().sum()
after = partner_clean[partner.columns].isnull().sum()
display(pd.DataFrame({"before": before, "after": after}))

print("\n[처리 후 customer_age 범위]:",
      partner_clean["customer_age"].min(), "~", partner_clean["customer_age"].max())
print("[amount_outlier=1 건수]:", partner_clean["amount_outlier"].sum())

[처리 전 후 결측 비교]


,before,after
order_id,0,0
customer_age,0,0
category,0,0
channel,0,0
price,5,0
quantity,0,0
amount,51,0
return_amount,0,0



[처리 후 customer_age 범위]: 5 ~ 60
[amount_outlier=1 건수]: 145


## 데이터 정제 결과

- `customer_age`의 비현실적인 값(0세, 999세)은 결측치로 변환한 후 중앙값으로 대체하였다.
- `quantity`의 이상치는 중앙값으로 대체하였다.
- `amount`의 극단값은 실제 거래 가능성을 고려하여 제거하지 않고 이상치 플래그(`amount_outlier`)를 추가하였다.
- `amount`의 결측치는 채널별 중앙값으로 대체하였다.
- `price`의 결측치는 상품 카테고리별 중앙값으로 대체하였다.

In [45]:
# 코드 퀴즈 — partner 데이터에 맞춘 답안

out = partner.copy()

# 1) quantity 이상치 → NaN
mask_q, _ = detect_outliers_iqr(out["quantity"])
out.loc[mask_q, "quantity"] = np.nan

# quantity 결측값 → 중앙값 대체
out["quantity"] = out["quantity"].fillna(
    out["quantity"].median()
).astype(int)

# 2) amount 결측 → 채널별 중앙값 대체
out["amount"] = out["amount"].fillna(
    out.groupby("channel")["amount"].transform("median")
)

# 3) 검증
print("quantity 결측:", out["quantity"].isnull().sum())
print("amount 결측:", out["amount"].isnull().sum())

quantity 결측: 0
amount 결측: 0


In [46]:
display(out[["quantity", "amount"]].head())

,quantity,amount
0,2,59800.0
1,3,389700.0
2,2,99800.0
3,3,149700.0
4,1,129900.0


In [47]:
display(pd.DataFrame({
    "처리 전 결측": partner[["quantity", "amount"]].isnull().sum(),
    "처리 후 결측": out[["quantity", "amount"]].isnull().sum()
}))

,처리 전 결측,처리 후 결측
quantity,0,0
amount,51,0


## 처리 결과

- `quantity`의 이상치를 결측치로 변환한 후 중앙값으로 대체하였다.
- `amount`의 결측치는 채널별 중앙값으로 대체하였다.
- 처리 후 두 변수 모두 남아 있는 결측치가 0개임을 확인하였다.

## 학습 메모

이번 파트를 진행하면서 결측치와 이상치의 종류, 처리 방법 등 이론적인 내용은 어느 정도 이해했다.

다만 아직은 데이터를 처음 봤을 때 어떤 문제를 먼저 찾아야 하는지, 어떤 처리 방법을 선택해야 하는지를 바로 판단하는 것은 어렵게 느껴진다.

또한 코드 한 줄 한 줄의 의미를 이해하는 데 시간이 많이 걸린다. 결과만 놓고 보면 어느 정도 해석할 수 있을 것 같은데, 코드와 결과를 연결해서 읽고 왜 이런 결과가 나왔는지를 이해하는 과정이 아직 오래 걸린다.

현재는 진도가 많이 밀려 있는 상태라 조급한 마음도 든다. 하지만 코드를 많이 읽고 직접 실행해 보면서 데이터 분석 과정을 반복하다 보면 점점 처리 속도와 이해도가 함께 좋아질 것이라고 생각한다.

앞으로는 단순히 코드를 따라 치는 것보다 **"왜 이 코드를 사용하는지"**, **"이 결과를 보고 무엇을 판단할 수 있는지"**를 스스로 설명하는 연습을 더 많이 해야겠다.
에이아이 도움 없이 혼자 정보 처리하는 것이 아직 어렵다.